In [ ]:
!pip install llama-index graspologic numpy==1.24.4 scipy==1.12.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of graspologic to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 60.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━

**Load Data**
1. sample news article dataset
2. 2,500 samples; for ease of experimentation, we will use 5 of these samples, which include the title and text of news articles.

In [ ]:
import pandas as pd
from llama_index.core import Document

book = pd.read_csv(
    "https://raw.githubusercontent.com/SakshiRawat13/dataset/refs/heads/main/output_chunks.csv"
)

book.head()

,Chunk Number,Content
0,Chunk 1,Metamorphosis is a book by Franz Kafka. Transl...
1,Chunk 2,He’d fall right\noff his desk! And it’s a funn...
2,Chunk 3,But this short\nconversation made the other me...
3,Chunk 4,And he could not knock himself out now at any ...
4,Chunk 5,Why did Gregor have to be the\nonly one condem...


**Document Object:** Each piece of information is encapsulated in a Document object that contains metadata, relationships, and the main content. The object has the following fields:

**id_:** A unique identifier for the document (e.g., 'a8bbf27f-e764-488d-89d6-36dd92bedac9'). This helps in tracking and referencing specific documents.

**embedding:** Indicates any associated embeddings for semantic analysis (currently None, meaning embeddings are not generated or included).

**metadata:** An empty dictionary {} representing metadata associated with the document, which can store information such as the source, author, or publication date.

**excluded_embed_metadata_keys:** A list of metadata keys that should be excluded from embedding operations. It’s empty [], suggesting that all metadata (if present) would be eligible for embedding.

**excluded_llm_metadata_keys:** A list of metadata keys to be excluded from processing by large language models (also empty []).

**relationships:** Represents any relational connections to other documents or entities (currently {}, meaning no relationships are defined).

**text:** The main content of the document, which can vary in length and detail. It’s formatted as a plain string.

**mimetype:** Specifies the type of content, which is 'text/plain' in all cases, indicating simple textual data.

**start_char_idx and end_char_idx:** Indices marking the start and end of the text content (currently None, suggesting that the entire text is used without specific substring indexing).

**text_template:** A template string indicating how to format text with metadata (e.g., '{metadata_str}\n\n{content}'). This field defines how the document content should be presented, with metadata followed by the content.

**metadata_template:** Describes how to format individual metadata items. It uses the pattern '{key}: {value}'.

**metadata_seperator:** Specifies the separator to use between metadata items, which is a newline character '\n'.


In [ ]:
documents = [
    Document(text=f"{row['Content']}")
    for i, row in book.iterrows()
]

print(documents)

[Document(id_='c0d524c6-7150-40b0-8723-9ec52a95a60d', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Metamorphosis is a book by Franz Kafka. Translated by David Wyllie.\nI\nOne morning, when Gregor Samsa woke from troubled dreams, he found\nhimself transformed in his bed into a horrible vermin. He lay on his\narmour-like back, and if he lifted his head a little he could see his\nbrown belly, slightly domed and divided by arches into stiff sections.\nThe bedding was hardly able to cover it and seemed ready to slide off\nany moment. His many legs, pitifully thin compared with the size of the\nrest of him, waved about helplessly as he looked.\n\n“What’s happened to me?” he thought. It wasn’t a dream. His room, a\nproper human room although a little too small, lay peacefully between\nits four familiar walls. A collection of textile samples lay spread out\non th

**Setup API Key and LLM**

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4")

1. **GraphRAGExtractor** is a class derived from **TransformComponent,** which is part of a **framework for knowledge graph extraction**.
The class extracts **triples (subject-relation-object)** from text using a language model (LLM). It can also **process and add descriptions for entities(subject and object) and relationships**.

Why is metadata.copy() Called Twice?
metadata = node.metadata.copy() is called twice to manage metadata for two different purposes:
First Call (For Entity Nodes): The metadata is copied before iterating over entities. This allows you to modify the **metadata specifically for entity nodes by adding "entity_description"** to it. The modified metadata is then used to **create EntityNode objects.**
Second Call (For Relationship Nodes): The metadata is copied again before iterating over entities_relationship. This **separate copy is modified for relationships, where "relationship_description" is added**. This ensures that any changes made to the metadata for entities do not affect the metadata used for relationships.

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from typing import Any, List, Callable, Optional, Union, Dict
from IPython.display import Markdown, display

from llama_index.core.async_utils import run_jobs
from llama_index.core.indices.property_graph.utils import (
    default_parse_triplets_fn,
)
from llama_index.core.graph_stores.types import (
    EntityNode,
    KG_NODES_KEY,
    KG_RELATIONS_KEY,
    Relation,
)
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.default_prompts import (
    DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
)
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.bridge.pydantic import BaseModel, Field

class GraphRAGExtractor(TransformComponent):
    """Extract triples from a graph.

    Uses an LLM and a simple prompt + output parsing to extract paths (i.e. triples) and entity, relation descriptions from text.

    Args:
        llm (LLM):
            The language model to use.
        extract_prompt (Union[str, PromptTemplate]):
            The prompt to use for extracting triples.
        parse_fn (callable):
            A function to parse the output of the language model.
        num_workers (int):
            The number of workers to use for parallel processing.
        max_paths_per_chunk (int):
            The maximum number of paths to extract per chunk.
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int #it determines how many pieces of text can be processed in parallel, which helps speed up the overall extraction process.
    max_paths_per_chunk: int #max_paths_per_chunk refers to the maximum number of triples that the system should extract from a single chunk of text.

    def __init__(
        self,
        llm: Optional[LLM] = None, #if we don't have an llm provided then the value is none
        extract_prompt: Optional[Union[str, PromptTemplate]] = None, #string or prompttemplate is given, string(if given) is converted to prompttemplate
        parse_fn: Callable = default_parse_triplets_fn,
        max_paths_per_chunk: int = 10,
        num_workers: int = 4,
    ) -> None:
        """Init params."""
        from llama_index.core import Settings

        if isinstance(extract_prompt, str): #If a string is passed as the extraction prompt, it converts it into a PromptTemplate object.
            extract_prompt = PromptTemplate(extract_prompt)

        super().__init__(
            llm=llm or Settings.llm, #if llm value is none default llm from settings is used
            extract_prompt=extract_prompt or DEFAULT_KG_TRIPLET_EXTRACT_PROMPT, #if no extract prompt given then default_kg_triplet_extract_prompt used
            parse_fn=parse_fn,
            num_workers=num_workers,
            max_paths_per_chunk=max_paths_per_chunk,
        )

    @classmethod
    def class_name(cls) -> str:
        return "GraphExtractor"

    def __call__(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]: #basenode is part of llama index, specific type of node this is
        """Extract triples from nodes."""
        return asyncio.run(
            self.acall(nodes, show_progress=show_progress, **kwargs)
        )

    async def _aextract(self, node: BaseNode) -> BaseNode: #asynchronous method that extracts triples from a BaseNode object,
    #The function returns a modified BaseNode object with extracted entities and relationships stored in its metadata.
        """Extract triples from a node."""
        assert hasattr(node, "text") #Ensures the node object has a text attribute. If it doesn’t, the function raises an AssertionError.

        text = node.get_content(metadata_mode="llm") #extract its content in a format suitable for language model processing.

        print("Text from node:", text)
        print("Initial metadata:", node.metadata)

        try:
            llm_response = await self.llm.apredict(
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities, entities_relationship = self.parse_fn(llm_response)
            print("Extracted Entities:", entities)
            print("Extracted Relationships:", entities_relationship)
        except ValueError:
            entities = []
            entities_relationship = []

        #Retrieves and removes existing entity nodes and relationships from node.metadata. If these keys don't exist, empty lists are returned.
        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        print("Existing KG Nodes:", existing_nodes)
        print("Existing KG Relations:", existing_relations)

        #Makes a copy of node.metadata to be used when creating new nodes and relations.
        metadata = node.metadata.copy()
        print("Metadata copied for entity modification:", metadata)
        for entity, entity_type, description in entities:
            metadata[
                "entity_description"
            ] = description  # Not used in the current implementation. But will be useful in future work.
            entity_node = EntityNode(
                name=entity, label=entity_type, properties=metadata
            )
            existing_nodes.append(entity_node)

        # Metadata after entity modification
        print("Metadata after entity modification:", metadata)
        print("KG Nodes after adding entities:", existing_nodes)

        metadata = node.metadata.copy() #why call second time? this time we copy node's metadata to modify it accoridng to our requirements for relationship specifically, previously metadata was modified for entity
        print("Metadata copied for relationship modification:", metadata)
        for triple in entities_relationship:
            subj, rel, obj, description = triple
            subj_node = EntityNode(name=subj, properties=metadata)
            obj_node = EntityNode(name=obj, properties=metadata)
            metadata["relationship_description"] = description
            rel_node = Relation(
                label=rel,
                source_id=subj_node.id,
                target_id=obj_node.id,
                properties=metadata,
            )

            existing_nodes.extend([subj_node, obj_node]) #same as append(append adds 1 element, extend adds more than 1)
            existing_relations.append(rel_node)

        print("Final Metadata before updating node:", metadata)
        print("Final KG Nodes:", existing_nodes)
        print("Final KG Relations:", existing_relations)

        #one question unable to find answer to what if entity_node in existing_nodes.append(entity_node) and obj_node/subj_node in existing_nodes.extend([subj_node, obj_node]) is same
        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        print("Node metadata after update:", node.metadata)

        return node

    async def acall(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes async."""
        jobs = [] # jobs to store tasks (coroutines) that will be run concurrently.
        for node in nodes:
            jobs.append(self._aextract(node))

        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting paths from text",
        )


In [ ]:
import re
from llama_index.core.graph_stores import SimplePropertyGraphStore
import networkx as nx
from networkx.algorithms.community import girvan_newman

from llama_index.core.llms import ChatMessage

#Declares GraphRAGStore as a subclass of SimplePropertyGraphStore, inheriting its properties and methods.
class GraphRAGStore(SimplePropertyGraphStore):
    community_summary = {} #empty dictionary to store summaries for different graph communities.
    max_cluster_size = 5

    def generate_community_summary(self, text): #text is the input string containing relationships to summarize.
        """Generate summary for a given text using an LLM."""
        messages = [ #A list of ChatMessage objects. The first message provides instructions to the language model, and the second message contains the text to summarize.
            ChatMessage(
                role="system",
                content=(
                    "You are provided with a set of relationships from a knowledge graph, each represented as "
                    "entity1->entity2->relation->relationship_description. Your task is to create a summary of these "
                    "relationships. The summary should include the names of the entities involved and a concise synthesis "
                    "of the relationship descriptions. The goal is to capture the most critical and relevant details that "
                    "highlight the nature and significance of each relationship. Ensure that the summary is coherent and "
                    "integrates the information in a way that emphasizes the key aspects of the relationships."
                ),
            ),
            ChatMessage(role="user", content=text),
        ]
        response = OpenAI().chat(messages)
        clean_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return clean_response

    def build_communities(self):
      """Builds communities from the graph and summarizes them using Girvan–Newman."""
      nx_graph = self._create_nx_graph()

      # Apply the Girvan–Newman algorithm to detect communities
      communities_generator = girvan_newman(nx_graph)

      # Get the first partition (this produces the smallest communities first)
      communities = tuple(sorted(c) for c in next(communities_generator))

      # Convert the communities into a format compatible with _collect_community_info
      community_hierarchical_clusters = [
          {'node': node, 'cluster': idx} for idx, community in enumerate(communities) for node in community
      ]

      print(f"Detected communities: {community_hierarchical_clusters}")

      community_info = self._collect_community_info(nx_graph, community_hierarchical_clusters)
      print(f"Community info collected: {community_info}")

      self._summarize_communities(community_info)

    def _collect_community_info(self, nx_graph, clusters):
        """Collect detailed information for each node based on their community."""
        community_mapping = {item['node']: item['cluster'] for item in clusters}
        print(f"Community mapping: {community_mapping}")  # Print community mapping

        community_info = {}  # Initializes an empty dictionary to hold community-specific relationship details
        for item in clusters:
            cluster_id = item['cluster']
            node = item['node']
            if cluster_id not in community_info:
                community_info[cluster_id] = []  # Ensure each community cluster has a list to store relationship details

            for neighbor in nx_graph.neighbors(node):  # Find nodes connected to `node` within the same community
                if community_mapping.get(neighbor) == cluster_id:  # Check if the neighbor belongs to the same community
                    edge_data = nx_graph.get_edge_data(node, neighbor)  # Retrieves the data associated with the edge
                    if edge_data:
                        detail = f"{node} -> {neighbor} -> {edge_data['relationship']} -> {edge_data['description']}"
                        community_info[cluster_id].append(detail)
        # Filter out empty communities
        filtered_community_info = {k: v for k, v in community_info.items() if v}

        # Re-index the communities to have sequential numbering
        reindexed_community_info = {}
        for new_id, (old_id, details) in enumerate(filtered_community_info.items()):
            reindexed_community_info[new_id] = details

        return reindexed_community_info


    def _create_nx_graph(self):
        """Converts internal graph representation to NetworkX graph."""
        nx_graph = nx.Graph()  #Creates an empty NetworkX graph.
        for node in self.graph.nodes.values(): #Iterates over all nodes in the internal graph representation.
            nx_graph.add_node(str(node)) #Adds each node to the NetworkX graph.
        for relation in self.graph.relations.values(): #Iterates over all relationships in the graph.
            nx_graph.add_edge( #Adds an edge between two nodes with metadata like relationship and description.
                relation.source_id,
                relation.target_id,
                relationship=relation.label,
                description=relation.properties["relationship_description"],
            )
        return nx_graph


    def _summarize_communities(self, community_info):
        """Generate and store summaries for each community."""
        for community_id, details in community_info.items():
            details_text = (
                "\n".join(details) + "."
            )  # Ensure it ends with a period
            print(f"Summarizing community {community_id} with details: {details_text}")  # Print details before summarizing

            self.community_summary[
                community_id
            ] = self.generate_community_summary(details_text)
            print(f"Summary for community {community_id}: {self.community_summary[community_id]}")  # Print the summary generated for each community

    def get_community_summaries(self):
        """Returns the community summaries, building them if not already done."""
        if not self.community_summary:
            self.build_communities()
        print(f"Final community summaries: {self.community_summary}")  # Print final summaries
        return self.community_summary

In [ ]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.llms import LLM


class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        """Process all community summaries to generate answers to a specific query."""
        community_summaries = self.graph_store.get_community_summaries() #Fetches summaries of all communities from GraphRAGStore.
        print(f"Community summaries retrieved: {community_summaries}")  # Print community summaries

        community_answers = [
            self.generate_answer_from_summary(community_summary, query_str)
            for _, community_summary in community_summaries.items()
        ] #Loops through each community summary and uses generate_answer_from_summary to create a response specific to the query for each summary.

        print(f"Individual answers from community summaries: {community_answers}")  # Print individual answers

        final_answer = self.aggregate_answers(community_answers) #Combines all individual answers into one coherent final answer using the aggregate_answers method.
        return final_answer

    def generate_answer_from_summary(self, community_summary, query): #Basic prompting, Generates an answer to the given query using the LLM, based on a community summary.
        """Generate an answer from a community summary based on a given query using LLM."""
        prompt = (
            f"Given the community summary: {community_summary}, "
            f"how would you answer the following query? Query: {query}"
        )
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content="I need an answer based on the above information.",
            ),
        ]
        response = self.llm.chat(messages)
        cleaned_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return cleaned_response

    def aggregate_answers(self, community_answers):#Basic prompting, Combines multiple answers from different community summaries into one cohesive response using the LLM.
        """Aggregate individual community answers into a final, coherent response."""
        # intermediate_text = " ".join(community_answers)
        prompt = "Combine the following intermediate answers into a final, concise response."
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content=f"Intermediate answers: {community_answers}",
            ),
        ]
        final_response = self.llm.chat(messages)
        cleaned_final_response = re.sub(
            r"^assistant:\s*", "", str(final_response)
        ).strip()
        return cleaned_final_response

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter()
nodes = parser.get_nodes_from_documents(documents)

print(type(nodes))
print(len(nodes))

for i in range(min(35, len(nodes))):
    print(f"Node {i}: {nodes[i]}")

<class 'list'>
35
Node 0: Node ID: 990a4903-a74b-4293-88ec-330ff2fda3d7
Text: Metamorphosis is a book by Franz Kafka. Translated by David
Wyllie. I One morning, when Gregor Samsa woke from troubled dreams, he
found himself transformed in his bed into a horrible vermin. He lay on
his armour-like back, and if he lifted his head a little he could see
his brown belly, slightly domed and divided by arches into stiff
sections. ...
Node 1: Node ID: 70dff60c-c4f4-46b8-910b-a8b721ebd610
Text: He’d fall right off his desk! And it’s a funny sort of business
to be sitting up there at your desk, talking down at your subordinates
from up there, especially when you have to go right up close because
the boss is hard of hearing. Well, there’s still some hope; once I’ve
got the money together to pay off my parents’ debt to him—another five
or...
Node 2: Node ID: 701149f1-ab3c-4a47-bd3c-b3997b917b46
Text: But this short conversation made the other members of the family
aware that Gregor, against their ex

In [ ]:
KG_TRIPLET_EXTRACT_TMPL = """
-Goal-
Given a text document, identify all entities and their entity types from the text and all relationships among the identified entities.
Given the text, extract up to {max_knowledge_triplets} entity-relation triplets.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: Type of the entity
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relation: relationship between source_entity and target_entity
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other

Format each relationship as ("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)

3. When finished, output.

-Real Data-
######################
text: {text}
######################
output:"""

In [ ]:
entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'


def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships


kg_extractor = GraphRAGExtractor(
    llm=llm,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=2,
    parse_fn=parse_fn,
)


In [ ]:
from llama_index.core import PropertyGraphIndex
import pickle


index = PropertyGraphIndex(
    nodes=nodes,
    property_graph_store=GraphRAGStore(),
    kg_extractors=[kg_extractor],
    show_progress=True,
)

# Storing the PropertyGraphIndex object
with open("property_graph_index.pkl", "wb") as f:
    pickle.dump(index, f)
print("PropertyGraphIndex has been saved to 'property_graph_index.pkl'.")

# Printing nodes and relationships with descriptive messages

print("This is the full list of nodes:")
print(index.property_graph_store.graph.nodes)

print("This is the length of the full list of nodes:")
print(len(index.property_graph_store.graph.nodes))

# print("\nThis is the full list of relationships:")
# print(index.property_graph_store.graph.relations)

print("\nThese are the values of nodes:")
print(index.property_graph_store.graph.nodes.values())

# print("\nThese are the values of relationships:")
# print(index.property_graph_store.graph.relations.values())

print("\nThis is the first node value:")
print(list(index.property_graph_store.graph.nodes.values())[0])

# print("\nThis is the first relationship value:")
# print(list(index.property_graph_store.graph.relations.values())[0])

# print("\nThis is the 'relationship_description' of the first relationship:")
# print(list(index.property_graph_store.graph.relations.values())[0].properties["relationship_description"])



Extracting paths from text:   0%|          | 0/35 [00:00<?, ?it/s]

Text from node: He’d fall right
off his desk! And it’s a funny sort of business to be sitting up there
at your desk, talking down at your subordinates from up there,
especially when you have to go right up close because the boss is hard
of hearing. Well, there’s still some hope; once I’ve got the money
together to pay off my parents’ debt to him—another five or six years I
suppose—that’s definitely what I’ll do. That’s when I’ll make the big
change. First of all though, I’ve got to get up, my train leaves at
five.”

And he looked over at the alarm clock, ticking on the chest of drawers.
“God in Heaven!” he thought. It was half past six and the hands were
quietly moving forwards, it was even later than half past, more like
quarter to seven. Had the alarm clock not rung? He could see from the
bed that it had been set for four o’clock as it should have been; it
certainly must have rung. Yes, but was it possible to quietly sleep
through that furniture-rattling noise? True, he had not slept

Extracting paths from text:   3%|▎         | 1/35 [00:14<08:01, 14.17s/it]

Extracted Entities: [('Gregor', 'Person', 'Gregor is a member of a family who seems to be suffering from a serious cold. He is a travelling salesman by profession and has a habit of locking all doors at night. He is currently at home and is experiencing difficulty in moving due to his condition.'), ("Gregor's Father", 'Person', "Gregor's father is another member of the family who is concerned about Gregor's well-being. He knocks on Gregor's door and calls out to him, asking what's wrong."), ("Gregor's Sister", 'Person', "Gregor's sister is also a member of the family who is worried about Gregor. She pleads with Gregor to open the door and asks if he needs anything.")]
Extracted Relationships: [('Gregor', "Gregor's Father", 'Family', "Gregor and his father are related as they are part of the same family. His father shows concern for Gregor's well-being, indicating a familial relationship."), ('Gregor', "Gregor's Sister", 'Family', "Gregor and his sister are related as they are part of t

Extracting paths from text:   6%|▌         | 2/35 [00:16<04:03,  7.37s/it]

...

Extracted Entities: []
Extracted Relationships: []
Existing KG Nodes: []
Existing KG Relations: []
Metadata copied for entity modification: {}
Metadata after entity modification: {}
KG Nodes after adding entities: []
Metadata copied for relationship modification: {}
Final Metadata before updating node: {}
Final KG Nodes: []
Final KG Relations: []
Node metadata after update: {'nodes': [], 'relations': []}


Extracting paths from text: 100%|██████████| 35/35 [02:25<00:00,  4.14s/it]


Extracted Entities: [('Mr. Samsa', 'Person', 'A character in the story who is a part of the Samsa family. He is the husband of Mrs. Samsa and the father of Grete. He is seen interacting with the cleaner and his family members.'), ('Mrs. Samsa', 'Person', 'A character in the story who is a part of the Samsa family. She is the wife of Mr. Samsa and the mother of Grete. She is seen writing a letter to her contractor and interacting with the cleaner and her family members.'), ('Grete', 'Person', 'A character in the story who is a part of the Samsa family. She is the daughter of Mr. and Mrs. Samsa. She is seen writing a letter to her principal and interacting with the cleaner and her family members.'), ('The Cleaner', 'Person', 'A character in the story who works for the Samsa family. She is seen interacting with the Samsa family and leaving after finishing her work.'), ('The Flat', 'Location', 'The place where the Samsa family lives. It is mentioned multiple times in the story as the place

Generating embeddings: 100%|██████████| 4/4 [00:01<00:00,  3.90it/s]


PropertyGraphIndex has been saved to 'property_graph_index.pkl'.
This is the full list of nodes:
{'990a4903-a74b-4293-88ec-330ff2fda3d7': ChunkNode(label='text_chunk', embedding=[0.014781380072236061, -0.0049088383093476295, 0.006803307682275772, -0.00011084017751272768, 0.008184624835848808, 0.03333233669400215, -0.021881606429815292, -0.02656775526702404, -0.03111189976334572, -0.01892532967031002, 0.02291436679661274, 0.033048324286937714, 0.008107167668640614, -0.028375085443258286, 0.007719882298260927, -0.002956276061013341, 0.03888342157006264, 0.008462178520858288, -0.014407004229724407, -0.039477258920669556, -0.009352934546768665, 0.001506377593614161, 0.0068162172101438046, -0.008320174179971218, -0.021545959636569023, 0.017505284398794174, 0.031731557101011276, -0.016614530235528946, 0.016769442707300186, -0.0037824842147529125, -0.005576904863119125, -0.018150759860873222, -0.029149655252695084, -0.0041762241162359715, -0.012528671883046627, -0.021713782101869583, 0.010353

In [ ]:
# Loading the PropertyGraphIndex object
with open("property_graph_index.pkl", "rb") as f:
    loaded_index = pickle.load(f)
print("PropertyGraphIndex has been loaded from 'property_graph_index.pkl'.")

PropertyGraphIndex has been loaded from 'property_graph_index.pkl'.


In [ ]:
list(loaded_index.property_graph_store.graph.nodes.values())[-1]

EntityNode(label='entity', embedding=None, properties={'triplet_source_id': '26345d23-6fa9-498e-95b6-2a5c22beee2b'}, name='Chose')

In [ ]:
list(loaded_index.property_graph_store.graph.relations.values())[0]

Relation(label='Franz Kafka', source_id='Metamorphosis', target_id='Written by', properties={'relationship_description': 'Franz Kafka is the author of the book Metamorphosis.', 'triplet_source_id': '990a4903-a74b-4293-88ec-330ff2fda3d7'})

In [ ]:
list(loaded_index.property_graph_store.graph.relations.values())[0].properties[
    "relationship_description"
]

'Franz Kafka is the author of the book Metamorphosis.'

In [ ]:
loaded_index.property_graph_store.build_communities()

Detected communities: [{'node': 'Metamorphosis is a book by Franz Kafka. Translated by David Wyllie.\nI\nOne morning, when Gregor Samsa woke from troubled dreams, he found\nhimself transformed in his bed into a horrible vermin. He lay on his\narmour-like back, and if he lifted his head a little he could see his\nbrown belly, slightly domed and divided by arches into stiff sections.\nThe bedding was hardly able to cover it and seemed ready to slide off\nany moment. His many legs, pitifully thin compared with the size of the\nrest of him, waved about helplessly as he looked.\n\n“What’s happened to me?” he thought. It wasn’t a dream. His room, a\nproper human room although a little too small, lay peacefully between\nits four familiar walls. A collection of textile samples lay spread out\non the table—Samsa was a travelling salesman—and above it there hung a\npicture that he had recently cut out of an illustrated magazine and\nhoused in a nice, gilded frame. It showed a lady fitted out wit

In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=loaded_index.property_graph_store, llm=llm
)

In [ ]:
response = query_engine.query(
    "What are the overarching themes and ideas presented throughout the book?"
)
display(Markdown(f"{response.response}"))

Final community summaries: {0: 'The book "Metamorphosis" features Gregor Samsa as the main character. It was written by Franz Kafka and translated by David Wyllie. Franz Kafka is the author of "Metamorphosis," while David Wyllie is the translator of the book. The relationship between these entities highlights the key roles they play in the creation and dissemination of this literary work.', 1: "In the relationships depicted in the knowledge graph, Gregor is at the center of various connections. He is shown to have a strong attachment to a picture of a lady dressed in fur, symbolizing his attempt to claim a sense of identity in his transformed state. Gregor's familial ties are highlighted through his role as a son to his concerned parents, a brother to his caring sister, and a provider for his family. There is a clear conflict between Gregor and his father, who blames him for the family's troubles, while Gregor's sister struggles with the idea of getting rid of him.\n\nProfessionally, G

The overarching themes and ideas presented throughout the book "Metamorphosis" include struggles with identity, family dynamics, societal expectations, and the consequences of transformation. These themes are explored through Gregor's transformation, his attachment to a picture of a lady dressed in fur, his roles within his family, and his professional relationships. The book also delves into themes of neglect, abandonment, aggression, sympathy, and the complexity of relationships, as seen in the strained relationship between Gregor and his father, and the father's dual nature as both an aggressor and a source of empathy. However, it should be noted that the provided summary does not give comprehensive information about all the overarching themes and ideas presented throughout the book.

In [ ]:
response = query_engine.query("What are the relationships and interactions between major characters or entities across the book?")
display(Markdown(f"{response.response}"))

Final community summaries: {0: 'The book "Metamorphosis" features Gregor Samsa as the main character. It was written by Franz Kafka and translated by David Wyllie. Franz Kafka is the author of "Metamorphosis," while David Wyllie is the translator of the book. The relationship between these entities highlights the key roles they play in the creation and dissemination of this literary work.', 1: "In the relationships depicted in the knowledge graph, Gregor is at the center of various connections. He is shown to have a strong attachment to a picture of a lady dressed in fur, symbolizing his attempt to claim a sense of identity in his transformed state. Gregor's familial ties are highlighted through his role as a son to his concerned parents, a brother to his caring sister, and a provider for his family. There is a clear conflict between Gregor and his father, who blames him for the family's troubles, while Gregor's sister struggles with the idea of getting rid of him.\n\nProfessionally, G

The major characters in "Metamorphosis" have complex relationships and interactions. Gregor, the central character, has a strained relationship with his father, who blames him for the family's troubles. Gregor's sister, Grete, cares for him but struggles with the idea of getting rid of him. Gregor's parents, Mr. and Mrs. Samsa, have a caring relationship, with Mrs. Samsa often persuading Mr. Samsa to go to bed. However, there is a neglectful relationship between Grete and her mother, as Grete abandons her mother in alarm. The father also has a dual nature, being both an attacker and a source of sympathy, adding complexity to the family dynamics. Gregor's father also acts as a landlord to three gentlemen, creating tension within the household. The parents are considering finding a suitable partner for Grete, indicating a caring and protective relationship. These relationships and interactions highlight the family dynamics, societal expectations, and the challenges faced by Gregor in his transformed state.

In [ ]:
response = query_engine.query(
    "What are the significant events or turning points in the book?"
)
display(Markdown(f"{response.response}"))

Final community summaries: {0: 'The book "Metamorphosis" features Gregor Samsa as the main character. It was written by Franz Kafka and translated by David Wyllie. Franz Kafka is the author of "Metamorphosis," while David Wyllie is the translator of the book. The relationship between these entities highlights the key roles they play in the creation and dissemination of this literary work.', 1: "In the relationships depicted in the knowledge graph, Gregor is at the center of various connections. He is shown to have a strong attachment to a picture of a lady dressed in fur, symbolizing his attempt to claim a sense of identity in his transformed state. Gregor's familial ties are highlighted through his role as a son to his concerned parents, a brother to his caring sister, and a provider for his family. There is a clear conflict between Gregor and his father, who blames him for the family's troubles, while Gregor's sister struggles with the idea of getting rid of him.\n\nProfessionally, G

Based on the provided summaries, significant events or turning points in the book "Metamorphosis" include Gregor's transformation into an insect, which drastically changes his identity and relationships. His absence from work and the subsequent reactions of his boss and the chief clerk, as well as the ongoing conflict with his father, who blames him for the family's troubles, are also significant. Gregor's dependence on his sister for food and care, his changing food preferences due to his condition, and the discovery of his death by the cleaner are other key events. Additionally, the father's attack on Gregor with apples and his sympathy towards his distressed daughter highlight the complex family dynamics. The father's attempt to evict Gregor and a tenant's decision to leave immediately indicate a strained tenancy situation. However, some summaries do not provide information on these significant events or turning points.